# 12 Locale eval
Run Spanish and degraded questions through the app, score them with the same graders, and re-check the judge in Spanish.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-12"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: the Spanish set and its English sources

In [ ]:
import json, re, logging
import pandas as pd
from kit.llm import call_llm
from kit.prices import cost

for name in ("httpx", "httpx2"):  # the MCP library turns on one log line per HTTP call; hide them
    logging.getLogger(name).setLevel(logging.WARNING)

sys.path.insert(0, str(ROOT / "evals"))
from gates import hard_failures  # the same hard gates as Chapter 8

es = [json.loads(line) for line in open(ROOT / "evals" / "golden_es.jsonl")]
source = {q["qid"]: q for q in map(json.loads, open(CORPUS / "questions.jsonl"))}
en = [{"description": c["description"].replace("es", "en"), "kind": "english",
       "vars": {**c["vars"], "question": source[c["source_qid"]]["question"]}} for c in es if c["kind"] == "translated"]
print(len(en), "English,", sum(c["kind"] == "translated" for c in es), "Spanish,",
      sum(c["kind"] == "degraded" for c in es), "degraded (typos, mixed language)")
pd.DataFrame([{"case": c["description"], "kind": c["kind"], "question": c["vars"]["question"]} for c in es]).iloc[[0, 5, 15, 17, 18]]

Step 3: run every case through the app, with the same graders

In [ ]:
from app.app import SYSTEM, TOP_K, answer, build_prompt
from mcp_server.server import search

def app_ask(question):
    r = answer(question, folder=CORPUS)
    return {"text": r["text"], "cited": r["cited"]}

def score(cases, ask, setup):
    rows = []
    for c in cases:
        out = ask(c["vars"]["question"])
        fails = hard_failures(c["vars"], out)
        rows.append({"setup": setup, "kind": c["kind"], "case": c["description"], "question": c["vars"]["question"],
                     "passed": not fails, "why": "; ".join(fails), "cited": out["cited"], "text": out["text"][:120]})
    return rows

base = pd.DataFrame(score(en, app_ask, "as is") + score(es, app_ask, "as is"))
print("Hard gate pass rate:")
print(base.groupby("kind", sort=False).passed.mean().round(2))

Step 4: trace one Spanish failure

In [ ]:
miss = base[(base.kind == "translated") & ~base.passed]
if len(miss):
    row = miss.iloc[0]
    print(f"{row.case}: {row.question}\nWhy it failed: {row.why}\nAnswer: {row.text}")
    print("Search hits for the Spanish question:", [h["id"] for h in search(row.question, 3, CORPUS)] or "none")
    print("The search matches English words, so a Spanish question finds the wrong pages or none.")
else:
    print("No Spanish failure to trace in this run.")

Step 5: one fix: translate for search, answer in the user's language

In [ ]:
TRANSLATE = "Translate this help question into English. Reply with the English question only."

def localized_ask(question):
    english = call_llm(f"Question: {question}", system=TRANSLATE, max_output_tokens=1000)["text"].strip()
    hits = search(english, TOP_K, CORPUS)
    if not hits:
        return {"text": "No help page matches that question.", "cited": []}
    r = call_llm(build_prompt(question, hits), system=SYSTEM + " Reply in the language of the user question.",
                 max_output_tokens=400)
    found = set(re.findall(r"qn-\d+", r["text"]))
    cited = [h["id"] for h in hits if h["id"] in found] or [h["id"] for h in hits]
    return {"text": r["text"], "cited": cited}

fixed = pd.DataFrame(score(es, localized_ask, "translate for search"))
table = pd.concat([base, fixed]).pivot_table(index="kind", columns="setup", values="passed", aggfunc="mean", sort=False)
print(table.round(2))

ES_WORDS = {"el", "la", "los", "las", "de", "que", "y", "en", "una", "para", "con", "por", "tu", "tus", "puedes", "nota", "notas"}
def looks_spanish(text):
    return sum(w in ES_WORDS for w in re.findall(r"[a-záéíóúñ]+", text.lower())) >= 3
print("\nAdvisory: answers that look Spanish:", round(fixed.text.map(looks_spanish).mean(), 2),
      "(offline the mock echoes the question, so this number means little; check it live)")

Step 6: re-check the judge on 10 Spanish hand labels

In [ ]:
# The same judge prompt as 08_judge_calibration. A judge you trust in English may not hold in Spanish.
JUDGE_SYSTEM = ("You grade answers from a help assistant. PASS only if the answer is correct and fully "
                "supported by the reference. Any wrong detail, invented feature or refusal is a FAIL. "
                "Reply with PASS or FAIL, a colon, then one short reason.")
labels = pd.read_csv(ROOT / "evals" / "labels_es.csv")

def judge(row):
    r = call_llm(f"Question: {row.question}\nReference: {row.reference}\nAnswer: {row.answer}\n"
                 "Reply with PASS or FAIL.", system=JUDGE_SYSTEM, max_output_tokens=800)
    verdict = re.match(r"\s*(PASS|FAIL)", r["text"].upper())
    return verdict.group(1).lower() if verdict else "unparsed"

labels["judge_label"] = [judge(row) for row in labels.itertuples()]
tp = ((labels.human_label == "pass") & (labels.judge_label == "pass")).sum()
tn = ((labels.human_label == "fail") & (labels.judge_label == "fail")).sum()
print(pd.crosstab(labels.human_label, labels.judge_label, rownames=["human"], colnames=["judge"]))
print({"true_positive_rate": round(float(tp / (labels.human_label == "pass").sum()), 2),
       "true_negative_rate": round(float(tn / (labels.human_label == "fail").sum()), 2),
       "agreement": round(float((labels.human_label == labels.judge_label).mean()), 2)})
print("Compare with your English numbers from 08. Disagreements:")
labels[labels.human_label != labels.judge_label][["id", "answer", "human_label", "judge_label", "note"]]